# 🌴 Nakhlah AI – Date Fruit Classification (v2 Production)
### Author: Abderrahmane
### Project: Computer Vision on Date Fruits
### Rebuilt: 2026-04-12

---

## 📋 What's New in v2

| Area | v1 (Original) | v2 (This notebook) |
|------|--------------|--------------------|
| **Datasets** | Single dataset (d1 only) | 3 datasets merged with prefix isolation |
| **Model** | EfficientNet-B0, backbone frozen | EfficientNet-B2, two-stage fine-tuning |
| **Augmentation** | Basic flip + rotation | RandAugment + MixUp + label smoothing |
| **Optimizer** | (not shown in v1) | AdamW + OneCycleLR scheduler |
| **Best model saving** | Saved at end only | Saved automatically on best val accuracy |
| **Early stopping** | ❌ | ✅ patience=7 |
| **Class weights** | ❌ | ✅ handles imbalance |
| **Evaluation** | Basic print loop | Confusion matrix + per-class F1 report |
| **Data split** | No safety checks | Idempotent, prefixed, length-guarded |
| **num_workers** | 0 (slow) | auto-detected |
| **pin_memory** | ❌ | ✅ when CUDA available |

## ⚙️ 0. Install & Imports

In [ ]:
# Install kagglehub if not present
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
print("✅ Dependencies OK")

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import shutil
import random
import json
import copy
import time
from pathlib import Path
from collections import Counter

# ── Numeric / Visual ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

# ── ML / DL ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from tqdm.auto import tqdm
import kagglehub

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥  Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 📂 1. Paths & Config

All paths live in one place. Change `BASE_DIR` only if you move the project.

In [ ]:
# ── Google Colab cache (comment out if running locally) ───────────────────────
os.environ["KAGGLEHUB_CACHE"] = "/content/data"

# ── Directory tree ────────────────────────────────────────────────────────────
BASE_DIR      = Path().resolve().parent          # project root (ai/)
DATASET_DIR   = BASE_DIR / "dataset"
RAW_DIR       = DATASET_DIR / "raw"              # merged, prefixed images
PROCESSED_DIR = DATASET_DIR / "processed"        # train / val / test splits
MODELS_DIR    = BASE_DIR / "models"
LOGS_DIR      = BASE_DIR / "logs"

train_dir = PROCESSED_DIR / "train"
val_dir   = PROCESSED_DIR / "val"
test_dir  = PROCESSED_DIR / "test"

for d in [RAW_DIR, train_dir, val_dir, test_dir, MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters (single source of truth) ─────────────────────────────────
CFG = dict(
    img_size        = 260,       # EfficientNet-B2 native resolution
    batch_size      = 32,
    num_workers     = min(4, os.cpu_count() or 1),
    # Stage-1: only train new classifier head (backbone frozen)
    stage1_epochs   = 5,
    stage1_lr       = 1e-3,
    # Stage-2: unfreeze full model, fine-tune with smaller LR
    stage2_epochs   = 25,
    stage2_lr       = 5e-4,
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    mixup_alpha     = 0.2,       # set to 0.0 to disable MixUp
    early_stop_patience = 7,
    model_name      = "nakhlah_model_v2",
)

print("📁 Directory tree ready")
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

## 📦 2. Multi-Dataset Download & Merge

### Why prefix matters
Three separate Kaggle datasets each contain classes named `Ajwa`, `Medjool`, etc.  
Without a prefix, images from different sources silently overwrite each other.  
With a prefix (`d1_ajwa`, `d2_ajwa`, `d3_ajwa`) the model sees domain variation  
while label ambiguity and data-leakage bugs are eliminated.

In [ ]:
def copy_dataset(src_root: str, dst_root: Path, prefix: str) -> None:
    """
    Walk every class folder inside src_root and copy images into
    dst_root/<prefix>_<class_name>/ to prevent collision across datasets.

    Args:
        src_root : path returned by kagglehub.dataset_download()
        dst_root : unified RAW_DIR
        prefix   : short tag identifying the source dataset (e.g. 'd1')
    """
    src_root = Path(src_root)
    for class_dir in sorted(src_root.iterdir()):
        if not class_dir.is_dir():
            continue

        # Normalise: lowercase, spaces → underscore, add prefix
        norm_class = f"{prefix}_{class_dir.name.lower().replace(' ', '_')}"
        target = dst_root / norm_class
        target.mkdir(parents=True, exist_ok=True)

        copied = 0
        for img_file in class_dir.iterdir():
            if img_file.is_file():
                dst = target / img_file.name
                if not dst.exists():           # idempotent: skip if already there
                    shutil.copy2(img_file, dst)
                    copied += 1

        print(f"  [{prefix}] {class_dir.name:20s} → {norm_class:35s} ({copied} files)")

In [ ]:
# ── Dataset registry ─────────────────────────────────────────────────────────
DATASETS = [
    ("wadhasnalhamdan/date-fruit-image-dataset-in-controlled-environment", "d1"),
    ("omarmejawel/combining-date-fruit-datasets-for-classification",       "d2"),
    ("mfarazf/datasetf",                                                   "d3"),
]

if not any(RAW_DIR.iterdir()):          # skip if already merged
    for ds_slug, prefix in DATASETS:
        print(f"\n⬇️  Downloading {ds_slug} …")
        cache_path = kagglehub.dataset_download(ds_slug)
        print(f"   Cached at: {cache_path}")
        copy_dataset(cache_path, RAW_DIR, prefix)
    print("\n✅ All datasets merged into RAW_DIR")
else:
    print("⚡ RAW_DIR not empty — skipping download (delete to re-merge)")

# Class summary
classes_in_raw = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
print(f"\n📊 {len(classes_in_raw)} classes in RAW_DIR:")
for c in classes_in_raw:
    n = len(list((RAW_DIR / c).iterdir()))
    print(f"   {c:40s}  {n:>5} images")

## ✂️ 3. Train / Val / Test Split (70 / 15 / 15)

- Idempotent — re-running the cell does nothing if the split already exists.  
- Guards against classes with fewer than 2 images.  
- Uses `shutil.copy2` (preserves metadata) instead of `shutil.copy`.

In [ ]:
def split_dataset(raw_dir: Path, train_dir: Path, val_dir: Path, test_dir: Path) -> None:
    """Stratified 70/15/15 split with idempotency guard."""

    # Idempotency: if train already has content, skip entirely
    if any(train_dir.iterdir()):
        print("⚡ Split already exists — skipping (delete processed/ to redo)")
        return

    split_stats = {}
    skipped     = []

    for class_dir in sorted(raw_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [f for f in class_dir.iterdir() if f.is_file()]

        if len(images) < 6:            # need enough for a meaningful split
            skipped.append(class_dir.name)
            continue

        train_imgs, temp_imgs = train_test_split(
            images, test_size=0.30, random_state=SEED
        )
        val_imgs, test_imgs = train_test_split(
            temp_imgs, test_size=0.50, random_state=SEED
        )

        for split_root, split_imgs in [
            (train_dir, train_imgs),
            (val_dir,   val_imgs),
            (test_dir,  test_imgs),
        ]:
            dst_class = split_root / class_dir.name
            dst_class.mkdir(parents=True, exist_ok=True)
            for img in split_imgs:
                shutil.copy2(img, dst_class / img.name)

        split_stats[class_dir.name] = {
            "train": len(train_imgs),
            "val":   len(val_imgs),
            "test":  len(test_imgs),
        }

    print(f"✅ Split complete — {len(split_stats)} classes processed")
    if skipped:
        print(f"⚠️  Skipped (too few images): {skipped}")

    # Print summary table
    df = pd.DataFrame(split_stats).T
    df["total"] = df.sum(axis=1)
    print(df.to_string())


split_dataset(RAW_DIR, train_dir, val_dir, test_dir)

## 🔬 4. EDA – Class Distribution

In [ ]:
def count_per_class(root: Path) -> dict:
    return {d.name: len(list(d.iterdir())) for d in sorted(root.iterdir()) if d.is_dir()}

train_counts = count_per_class(train_dir)
val_counts   = count_per_class(val_dir)
test_counts  = count_per_class(test_dir)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(train_counts))
w = 0.28
ax.bar(x - w, train_counts.values(), w, label="train", color="#4C72B0")
ax.bar(x,     val_counts.values(),   w, label="val",   color="#55A868")
ax.bar(x + w, test_counts.values(),  w, label="test",  color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels(list(train_counts.keys()), rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Image count")
ax.set_title("Class distribution across splits")
ax.legend()
plt.tight_layout()
plt.savefig(LOGS_DIR / "class_distribution.png", dpi=120)
plt.show()
print(f"\n📊 Train total: {sum(train_counts.values())}  |  "
      f"Val: {sum(val_counts.values())}  |  Test: {sum(test_counts.values())}")

## 🔄 5. Data Augmentation & DataLoaders

**Changes from v1:**
- Input size raised to 260 (EfficientNet-B2 native).
- `RandAugment` replaces manual flip/rotation/jitter — more diverse, less to tune.
- `WeightedRandomSampler` applied to training set to handle class imbalance.
- `pin_memory=True` when GPU available (faster host→device transfer).
- `num_workers` auto-detected.

In [ ]:
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]
SZ       = CFG["img_size"]

# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(SZ, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandAugment(num_ops=2, magnitude=9),  # replaces manual jitter/rotation
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(SZ + 32),
    transforms.CenterCrop(SZ),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

# ── Datasets ─────────────────────────────────────────────────────────────────
train_dataset = ImageFolder(train_dir, transform=train_transform)
val_dataset   = ImageFolder(val_dir,   transform=eval_transform)
test_dataset  = ImageFolder(test_dir,  transform=eval_transform)

NUM_CLASSES = len(train_dataset.classes)
CLASS_NAMES = train_dataset.classes
print(f"✅ {NUM_CLASSES} classes: {CLASS_NAMES}")

# ── Weighted sampler (handles class imbalance) ────────────────────────────────
targets      = [s[1] for s in train_dataset.samples]
class_counts = Counter(targets)
weights      = [1.0 / class_counts[t] for t in targets]
sampler      = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# ── DataLoaders ───────────────────────────────────────────────────────────────
pin = DEVICE.type == "cuda"
nw  = CFG["num_workers"]

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    sampler=sampler,         # replaces shuffle=True — weighted sampling
    num_workers=nw,
    pin_memory=pin,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=nw,
    pin_memory=pin,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=nw,
    pin_memory=pin,
)

print(f"\n📦 Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"   Batches per epoch: {len(train_loader)}")

## 🖼️ 6. Sample Grid Visualization

In [ ]:
def denorm(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor(IMG_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMG_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

imgs, lbls = next(iter(train_loader))
n_show = min(16, len(imgs))
cols   = 8
rows   = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
axes = axes.flatten()
for i in range(n_show):
    axes[i].imshow(denorm(imgs[i]))
    axes[i].set_title(CLASS_NAMES[lbls[i]], fontsize=7)
    axes[i].axis("off")
for ax in axes[n_show:]:
    ax.axis("off")
plt.suptitle("Augmented training samples (first batch)", fontsize=10)
plt.tight_layout()
plt.savefig(LOGS_DIR / "sample_grid.png", dpi=120)
plt.show()

## 🧠 7. Model – EfficientNet-B2 with Two-Stage Fine-Tuning

**Why B2 over B0?**  
EfficientNet-B2 (260 px) has ~9 M parameters vs B0's ~5 M, meaningfully higher  
top-1 accuracy on ImageNet, and still trains comfortably on a T4 GPU.

**Two-stage strategy:**
1. **Stage 1** – freeze backbone, train only new classifier head. Fast convergence, no risk of destroying pretrained features.
2. **Stage 2** – unfreeze everything, fine-tune end-to-end with a lower learning rate and OneCycleLR.

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    Load ImageNet-pretrained EfficientNet-B2 and replace the classifier
    head with a regularised head for num_classes.

    Args:
        num_classes     : number of output classes
        freeze_backbone : if True, only the new head is trainable
    Returns:
        model ready for .to(device)
    """
    weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
    model   = models.efficientnet_b2(weights=weights)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace classifier — add an extra Linear + BN + Dropout for better regularisation
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model


model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\n🔢 Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 🎲 8. MixUp Augmentation (optional)

MixUp linearly interpolates pairs of training samples and their labels.  
It acts as a strong regulariser and is particularly effective for fine-grained  
visual classification. Set `CFG['mixup_alpha'] = 0.0` to disable.

In [ ]:
def mixup_batch(images: torch.Tensor, labels: torch.Tensor, alpha: float = 0.2):
    """
    Apply MixUp to a batch.

    Returns:
        mixed_images : Tensor[B, C, H, W]
        labels_a     : original labels
        labels_b     : shuffled labels
        lam          : mixing coefficient (scalar)
    """
    if alpha <= 0.0:
        return images, labels, labels, 1.0

    lam     = np.random.beta(alpha, alpha)
    bsize   = images.size(0)
    idx     = torch.randperm(bsize, device=images.device)
    mixed   = lam * images + (1 - lam) * images[idx]
    return mixed, labels, labels[idx], lam


def mixup_criterion(criterion, preds, y_a, y_b, lam):
    """Compute MixUp loss as convex combination of two CE losses."""
    return lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)


print("✅ MixUp helpers defined  (alpha={:.2f})".format(CFG["mixup_alpha"]))

## 🔁 9. Training & Validation Loops

In [ ]:
def train_one_epoch(
    model, loader, criterion, optimizer, scheduler, device,
    mixup_alpha=0.2, scaler=None
):
    """
    One full training epoch with optional MixUp and AMP.

    Returns:
        avg_loss (float), accuracy (float %)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="  train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        images, labels_a, labels_b, lam = mixup_batch(images, labels, alpha=mixup_alpha)

        optimizer.zero_grad()

        if scaler is not None:   # AMP path
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:                    # standard path
            outputs = model(images)
            loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()         # OneCycleLR steps per batch

        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()  # approximate when MixUp active
        total      += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Full evaluation pass — no augmentation, no MixUp.

    Returns:
        avg_loss (float), accuracy (float %)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="  eval ", leave=False):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs        = model(images)
        loss           = criterion(outputs, labels)
        total_loss    += loss.item()
        correct       += (outputs.argmax(1) == labels).sum().item()
        total         += labels.size(0)

    return total_loss / len(loader), 100.0 * correct / total


print("✅ Training / evaluation functions defined")

## 🚀 10. Stage 1 – Train Classifier Head Only

Backbone frozen → fast convergence, learns task-specific features in the new head.

In [ ]:
# Loss with label smoothing reduces overconfidence
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"]).to(DEVICE)

# AMP scaler (only meaningful on CUDA)
scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

# ── Stage 1 optimiser ─────────────────────────────────────────────────────────
optimizer_s1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["stage1_lr"],
    weight_decay=CFG["weight_decay"],
)
scheduler_s1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_s1,
    max_lr=CFG["stage1_lr"],
    steps_per_epoch=len(train_loader),
    epochs=CFG["stage1_epochs"],
)

# ── History dicts ─────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

best_val_acc  = 0.0
best_ckpt     = MODELS_DIR / f"{CFG['model_name']}_best.pth"
patience_left = CFG["early_stop_patience"]

print(f"\n{'='*60}")
print(f"STAGE 1 — {CFG['stage1_epochs']} epochs, lr={CFG['stage1_lr']}")
print(f"  (backbone frozen, training classifier head only)")
print(f"{'='*60}\n")

for epoch in range(1, CFG["stage1_epochs"] + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_s1, scheduler_s1,
        DEVICE, mixup_alpha=CFG["mixup_alpha"], scaler=scaler
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(optimizer_s1.param_groups[0]["lr"])

    elapsed = time.time() - t0
    flag = ""

    # ── Save best model ───────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "val_acc":     val_acc,
            "class_names": CLASS_NAMES,
            "cfg":         CFG,
        }, best_ckpt)
        flag = " ★ best saved"
        patience_left = CFG["early_stop_patience"]  # reset patience
    else:
        patience_left -= 1

    print(
        f"Epoch {epoch:3d}/{CFG['stage1_epochs']}  "
        f"Train Loss={train_loss:.4f}  Acc={train_acc:6.2f}%  "
        f"Val Loss={val_loss:.4f}  Val Acc={val_acc:6.2f}%  "
        f"[{elapsed:.0f}s]{flag}"
    )

    if patience_left == 0:
        print("⏹  Early stopping triggered")
        break

print(f"\n✅ Stage 1 done. Best val acc: {best_val_acc:.2f}%")

## 🔓 11. Stage 2 – Unfreeze & Fine-Tune End-to-End

Unfreeze all backbone layers. Use a lower learning rate and reset the OneCycleLR  
scheduler so the entire model converges smoothly.

In [ ]:
# ── Unfreeze backbone ─────────────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = True

trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🔓 All params unfrozen — trainable: {trainable2:,}")

# ── Stage 2 optimiser ─────────────────────────────────────────────────────────
optimizer_s2 = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["stage2_lr"],
    weight_decay=CFG["weight_decay"],
)
scheduler_s2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_s2,
    max_lr=CFG["stage2_lr"],
    steps_per_epoch=len(train_loader),
    epochs=CFG["stage2_epochs"],
)

patience_left = CFG["early_stop_patience"]  # reset patience counter

print(f"\n{'='*60}")
print(f"STAGE 2 — {CFG['stage2_epochs']} epochs, max_lr={CFG['stage2_lr']}")
print(f"  (full model, end-to-end fine-tuning)")
print(f"{'='*60}\n")

for epoch in range(CFG["stage1_epochs"] + 1, CFG["stage1_epochs"] + CFG["stage2_epochs"] + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_s2, scheduler_s2,
        DEVICE, mixup_alpha=CFG["mixup_alpha"], scaler=scaler
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(optimizer_s2.param_groups[0]["lr"])

    elapsed = time.time() - t0
    flag = ""

    # ── Save best model ───────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "val_acc":     val_acc,
            "class_names": CLASS_NAMES,
            "cfg":         CFG,
        }, best_ckpt)
        flag = " ★ best saved"
        patience_left = CFG["early_stop_patience"]
    else:
        patience_left -= 1

    print(
        f"Epoch {epoch:3d}/{CFG['stage1_epochs'] + CFG['stage2_epochs']}  "
        f"Train Loss={train_loss:.4f}  Acc={train_acc:6.2f}%  "
        f"Val Loss={val_loss:.4f}  Val Acc={val_acc:6.2f}%  "
        f"[{elapsed:.0f}s]{flag}"
    )

    if patience_left == 0:
        print("⏹  Early stopping triggered")
        break

print(f"\n✅ Training complete. Best val acc: {best_val_acc:.2f}%")
print(f"   Best checkpoint: {best_ckpt}")

## 📈 12. Training Curves

In [ ]:
epochs_ran = len(history["train_loss"])
xs = range(1, epochs_ran + 1)
stage_boundary = CFG["stage1_epochs"]  # draw vertical line here

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(xs, history["train_loss"], label="Train")
axes[0].plot(xs, history["val_loss"],   label="Val")
axes[0].axvline(stage_boundary, color="gray", linestyle="--", alpha=0.6, label="S1→S2")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

# Accuracy
axes[1].plot(xs, history["train_acc"], label="Train")
axes[1].plot(xs, history["val_acc"],   label="Val")
axes[1].axvline(stage_boundary, color="gray", linestyle="--", alpha=0.6, label="S1→S2")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Epoch"); axes[1].legend()

# Learning rate
axes[2].semilogy(xs, history["lr"])
axes[2].axvline(stage_boundary, color="gray", linestyle="--", alpha=0.6, label="S1→S2")
axes[2].set_title("Learning Rate (log)"); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig(LOGS_DIR / "training_curves.png", dpi=120)
plt.show()

## 🧪 13. Test Set Evaluation

Load the best checkpoint (not the last epoch — important!) and run full  
evaluation including confusion matrix and per-class F1 report.

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"📂 Loaded best checkpoint from epoch {ckpt['epoch']} (val acc={ckpt['val_acc']:.2f}%)")

# ── Collect all predictions ───────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE, non_blocking=True)
        preds  = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# ── Metrics ───────────────────────────────────────────────────────────────────
test_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f"\n🎯 Test Accuracy: {test_acc:.2f}%")
print("\n" + classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    digits=3
))

## 🗺️ 14. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], xticks_rotation=45, colorbar=False
)
axes[0].set_title("Confusion Matrix (counts)")

# Normalised
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
    ax=axes[1], xticks_rotation=45, colorbar=False
)
axes[1].set_title("Confusion Matrix (normalised)")

plt.tight_layout()
plt.savefig(LOGS_DIR / "confusion_matrix.png", dpi=120)
plt.show()

## 💾 15. Save Final Model + Metadata

Two artefacts are saved:
- `nakhlah_model_v2_best.pth` — best validation checkpoint (saved during training).
- `nakhlah_model_v2_final.pth` — final epoch weights (useful for ensembling).
- `nakhlah_model_v2_metadata.json` — class names, config, and achieved accuracy.

In [ ]:
# ── Final weights ─────────────────────────────────────────────────────────────
final_ckpt = MODELS_DIR / f"{CFG['model_name']}_final.pth"
torch.save({
    "epoch":       epochs_ran,
    "model_state": model.state_dict(),
    "val_acc":     history["val_acc"][-1],
    "class_names": CLASS_NAMES,
    "cfg":         CFG,
}, final_ckpt)

# ── Metadata JSON ─────────────────────────────────────────────────────────────
metadata = {
    "model":       "efficientnet_b2",
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "best_val_acc": best_val_acc,
    "test_acc":     test_acc,
    "epochs_ran":   epochs_ran,
    "cfg":          CFG,
    "img_mean":     IMG_MEAN,
    "img_std":      IMG_STD,
}
meta_path = MODELS_DIR / f"{CFG['model_name']}_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))

print(f"✅ Final checkpoint → {final_ckpt}")
print(f"✅ Best checkpoint  → {best_ckpt}")
print(f"✅ Metadata         → {meta_path}")
print(f"\n🏆 Best Val Acc: {best_val_acc:.2f}%   |   Test Acc: {test_acc:.2f}%")

## 🔍 16. Inference Helper

Load the best model and classify a single image. Useful for deployment testing.

In [ ]:
def predict_image(image_path: str, ckpt_path: str = str(best_ckpt)) -> dict:
    """
    Run inference on a single image file.

    Args:
        image_path : path to an image file
        ckpt_path  : path to a .pth checkpoint

    Returns:
        dict with 'class', 'confidence', and top-5 'probabilities'
    """
    # ── Load checkpoint ───────────────────────────────────────────────────────
    ckpt       = torch.load(ckpt_path, map_location="cpu")
    cls_names  = ckpt["class_names"]
    n_cls      = len(cls_names)
    inf_model  = build_model(n_cls, freeze_backbone=False)
    inf_model.load_state_dict(ckpt["model_state"])
    inf_model.eval()

    # ── Preprocess ────────────────────────────────────────────────────────────
    tfm = transforms.Compose([
        transforms.Resize(ckpt["cfg"]["img_size"] + 32),
        transforms.CenterCrop(ckpt["cfg"]["img_size"]),
        transforms.ToTensor(),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])
    img    = Image.open(image_path).convert("RGB")
    tensor = tfm(img).unsqueeze(0)

    # ── Forward pass ──────────────────────────────────────────────────────────
    with torch.no_grad():
        probs = F.softmax(inf_model(tensor), dim=1).squeeze()

    top5_vals, top5_idx = probs.topk(min(5, n_cls))
    top5 = {cls_names[i]: round(v.item(), 4) for i, v in zip(top5_idx, top5_vals)}

    return {
        "class":       cls_names[top5_idx[0].item()],
        "confidence":  round(top5_vals[0].item() * 100, 2),
        "top5_probs":  top5,
    }


# ── Quick sanity-check: pick a random test image ──────────────────────────────
sample_class_dir = next(test_dir.iterdir())
sample_img       = next(sample_class_dir.iterdir())
result           = predict_image(str(sample_img))

print(f"🔍 Sample image  : {sample_img.name}")
print(f"   True class    : {sample_class_dir.name}")
print(f"   Predicted     : {result['class']} ({result['confidence']:.1f}%)")
print(f"   Top-5 probs   : {result['top5_probs']}")